# Checkup figures

Makes the figures for the Nov 4 checkup deck from the specs in [`docs/CHECKUP-VISUALS.md`](../docs/CHECKUP-VISUALS.md). Every figure is saved under `docs/figures/` with the cell that made it.

| Cell | Figure | Deck slot | Needs the data |
|---|---|---|---|
| A | per-class IoU, ours beside the paper, the gap, the 39.1 and 52.8 lines | slide 7 | no |
| B | architecture strip with status, lecture tie-ins, and the Phase 2 ablation brackets | slide 8 | no |
| C | one cube: RGB frame if the release has one, false colour with label outlines, mask, spectrum | slide 6 or 3 | yes |
| D | spectral shape per material, six small multiples, plus a raw-brightness version | slide 3 or 6 | yes |
| G | training curves from `metrics.csv` | Nov 5 | a metrics file |

Numbers in A come from `docs/RESULTS.md` (row `cnn_ds2`) and Casao et al., IROS 2024, Table IV. Shapes in B come from `src/sw/models/transformer.py`. Cells C and D read `labels_hyper_lt/train` and `hyper/train` under `SW_ROOT` and skip themselves if it is not set. D takes the 8 train cubes with the most pixels of each object class, so basket, tape, and filament are on the chart.

Class colours are the dataset palette from `meta.json` (`sw.data.PALETTE`), so the charts agree with the mask. Light colours (film, basket, tape, bag) are drawn with a dark halo, every bar has an ink edge and a printed value, and the small multiples in D identify a class by its panel, not by colour alone. Type is sized for a 16:9 slide: every figure is drawn on a 13.33 by 7.5 inch canvas (`SLIDE`), so a 13 pt label on the figure is about 13 pt on the slide.


## Colab setup

Skip on your own machine. On Colab it clones the repo and installs the libraries. The 23 GB download runs only when `DOWNLOAD_DATA` is True; cells A, B and G do not need it.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
DOWNLOAD_DATA = False  # True only for cells C and D (about 23 GB)
if IN_COLAB:
    os.chdir("/content")
    repo = Path("/content/255-final-project")
    if not repo.is_dir():
        subprocess.check_call(
            ["git", "clone", "https://github.com/scottn66/255-final-project.git", str(repo)]
        )
    os.chdir(repo)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
    if DOWNLOAD_DATA and not os.environ.get("SW_ROOT"):
        subprocess.check_call(["bash", "scripts/get_data.sh", "/content/data"])
        os.environ["SW_ROOT"] = "/content/data/spectralwaste_segmentation"

## Setup

Imports, the class palette, the output folder, and the nominal wavelength axis. `meta.json` ships no wavelengths, so the nm axis assumes even spacing of the 224 bands over 900 to 1700 nm and says so.

In [ ]:
import os
import sys
from pathlib import Path


def _find_repo_root():
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if (cand / "src" / "sw" / "__init__.py").is_file():
            return cand
    raise RuntimeError("Start Jupyter from the repo root or from the notebooks/ folder.")


REPO = _find_repo_root()
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import FancyBboxPatch, Patch

from sw.data import CLASS_NAMES, HYPER_LT_TRAIN_PIXEL_COUNTS, PALETTE, UINT16_MAX

FIG_DIR = REPO / "docs" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
SW_ROOT = os.environ.get("SW_ROOT")

OBJECT_CLASSES = CLASS_NAMES[1:]  # film .. bag, ids 1..6
COLOR = {name: np.array(rgb) / 255 for name, rgb in zip(CLASS_NAMES, PALETTE)}
LABEL = {**{name: name for name in CLASS_NAMES}, "video_tape": "tape"}
INK, MUTED, PAPER_FACE = "#222222", "#5c5c5c", "#f3f3f3"
PER_CLASS = 8  # richest train cubes per object class, cell D
SLIDE = (13.33, 7.5)  # 16:9, so a figure fills the slide and its type stays readable

N_BANDS, GROUP, N_GROUPS = 224, 16, 14
FALSE_COLOUR_BANDS = (56, 112, 168)  # notebook 01: C // 4, C // 2, 3C // 4
LINESTYLE = {
    "background": ":",
    "film": "-",
    "basket": "--",
    "cardboard": "-",
    "video_tape": "-.",
    "filament": "-",
    "bag": (0, (5, 1.6)),
}


def nominal_nm(band):
    """Nominal wavelength. meta.json ships none; assume even spacing over 900-1700 nm."""
    return 900 + np.asarray(band, dtype=float) * 800 / (N_BANDS - 1)


def band_of_nm(nm):
    return (np.asarray(nm, dtype=float) - 900) * (N_BANDS - 1) / 800


plt.rcParams.update(
    {
        "font.size": 15,
        "axes.titlesize": 16,
        "axes.labelsize": 15,
        "xtick.labelsize": 13,
        "ytick.labelsize": 14,
        "legend.fontsize": 13,
        "axes.linewidth": 0.8,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "hatch.linewidth": 0.6,
        "savefig.dpi": 200,
        "savefig.bbox": "tight",
        "savefig.pad_inches": 0.28,
        "figure.facecolor": "white",
        "axes.facecolor": "white",
    }
)


def save(fig, name, svg=False):
    """Write docs/figures/<name>.png (and .svg when asked) and print the path."""
    png = FIG_DIR / f"{name}.png"
    fig.savefig(png)
    if svg:
        fig.savefig(FIG_DIR / f"{name}.svg")
    print("saved", png.relative_to(REPO))
    plt.close(fig)
    return png


def train_dirs():
    """Label and cube folders for the train split. Labels may outnumber cubes."""
    root = Path(SW_ROOT)
    labels = root / "labels_hyper_lt" / "train"
    cubes = root / "hyper" / "train"
    if not labels.is_dir():
        raise FileNotFoundError(f"missing {labels}")
    return labels, cubes


def load_mask(path):
    from PIL import Image

    with Image.open(path) as img:
        mask = np.array(img)
    if mask.ndim != 2:
        raise ValueError(f"{path.name} has shape {mask.shape}; expected (H, W) class ids")
    return mask


def load_cube(path):
    import tifffile

    cube = np.asarray(tifffile.imread(path), dtype=np.float32) / np.float32(UINT16_MAX)
    if cube.ndim != 3 or cube.shape[0] != N_BANDS:
        raise ValueError(f"{path.name} has shape {cube.shape}; expected ({N_BANDS}, H, W)")
    return cube


def class_counts(mask):
    counts = np.zeros(len(CLASS_NAMES), dtype=np.int64)
    ids, n = np.unique(mask, return_counts=True)
    for i, k in zip(ids, n):
        if 0 <= int(i) < len(counts):
            counts[int(i)] = int(k)
    return counts


def choose_example_stem(label_dir):
    """Filament present, then the most classes, then the earliest stem."""
    best = None
    for index, path in enumerate(sorted(label_dir.glob("*.png"))):
        ids = np.unique(load_mask(path))
        key = (5 in set(ids.tolist()), int(len(ids)), -index)
        if best is None or key > best[0]:
            best = (key, path.stem)
    return best[1]


def choose_class_cubes(label_dir, per_class=PER_CLASS):
    """Up to `per_class` stems with the most pixels of each object class."""
    stems = sorted(p.stem for p in label_dir.glob("*.png"))
    counts = np.zeros((len(stems), len(CLASS_NAMES)), dtype=np.int64)
    for i, stem in enumerate(stems):
        counts[i] = class_counts(load_mask(label_dir / f"{stem}.png"))
    chosen = {}
    for c, name in enumerate(CLASS_NAMES):
        if name == "background":
            continue
        order = np.argsort(-counts[:, c], kind="stable")
        picked = []
        for i in order:
            if counts[i, c] <= 0:
                break
            picked.append(stems[int(i)])
            if len(picked) >= per_class:
                break
        chosen[name] = picked
    return chosen


def find_rgb(stem):
    """The release's RGB frame for a stem, if an rgb/train folder exists next to hyper/. None otherwise."""
    if not SW_ROOT:
        return None
    folder = Path(SW_ROOT) / "rgb" / "train"
    for ext in ("png", "jpg", "jpeg", "tiff", "tif"):
        path = folder / f"{stem}.{ext}"
        if path.is_file():
            return path
    return None


def shape_only(pix):
    """Divide each pixel's spectrum by its own mean, so brightness drops out and only the shape stays."""
    return pix / np.maximum(pix.mean(axis=0, keepdims=True), 1e-6)


def draw_line(ax, x, y, name, label=None, lw=2.3):
    """Palette colour with a dark halo so film, basket, tape, and bag stay visible."""
    color = "0.45" if name == "background" else COLOR[name]
    ls = LINESTYLE[name]
    if name == "background":
        ax.plot(x, y, color=color, lw=1.7, ls=ls, label=label, zorder=3)
        return
    ax.plot(x, y, color=INK, lw=lw + 1.35, ls=ls, solid_capstyle="round", zorder=3)
    ax.plot(x, y, color=color, lw=lw, ls=ls, solid_capstyle="round", label=label, zorder=4)


print("repo", REPO)
print("SW_ROOT", SW_ROOT or "(not set: cells C and D will skip)")


## A. Per-class IoU (slide 7)

Ours is val, half resolution for cube and mask, 30 epochs, best epoch 29, one run. The grey bars are the paper's MiniNet-v2 HYPER row, the fair comparison because MiniNetSeg is from the same family. Background (86.8) stays off the bars: it is not in the foreground mean.

Classes are sorted by the gap, paper minus ours, and the gap is printed after each pair. The two vertical lines are the headline numbers, 39.1 and 52.8, so the slide shows them rather than only stating them. The filament row carries the rare-class fact (0.098% of train pixels, loss weight 1.5x) because the meta-deck wants that row to hand off to the next-steps slide.


In [ ]:
# docs/RESULTS.md, row cnn_ds2, best val FG mIoU 0.391 at epoch 29, in percent
OURS = {"film": 55.2, "basket": 49.5, "cardboard": 57.8, "video_tape": 28.6, "filament": 0.0, "bag": 43.6}
OURS_MEAN = 39.1
# Casao et al., IROS 2024, Table IV, MiniNet-v2 HYPER row, in percent
PAPER = {"film": 61.2, "basket": 61.0, "cardboard": 78.8, "video_tape": 28.8, "filament": 30.5, "bag": 56.3}
PAPER_MEAN = 52.8
# src/sw/data.py: 33,101 of 33,685,504 train pixels; paper-scheme weight (median / f) ** 0.12
FILAMENT_NOTE = "filament is 0.098% of train pixels; its loss weight is only 1.5x"

assert abs(np.mean(list(OURS.values())) - OURS_MEAN) < 0.05
assert abs(np.mean(list(PAPER.values())) - PAPER_MEAN) < 0.05

gap = {c: PAPER[c] - OURS[c] for c in OBJECT_CLASSES}
order = sorted(OBJECT_CLASSES, key=lambda c: gap[c], reverse=True)
gap_share = (gap["filament"] + gap["cardboard"]) / sum(gap.values())
y = np.arange(len(order))[::-1]
h = 0.36
pad = dict(facecolor="white", edgecolor="none", pad=1.2)  # labels stay readable where they cross a line

fig, ax = plt.subplots(figsize=SLIDE)
ax.barh(y + h / 2, [OURS[c] for c in order], height=h, color=[COLOR[c] for c in order], edgecolor=INK, linewidth=0.9, zorder=2)
ax.barh(y - h / 2, [PAPER[c] for c in order], height=h, color=PAPER_FACE, edgecolor=INK, linewidth=0.9, zorder=2)
for yi, c in zip(y, order):
    if OURS[c] == 0:
        # A zero-width bar disappears, so mark the class colour at the axis.
        ax.plot([1.0, 1.0], [yi + 0.04, yi + h - 0.04], color=COLOR[c], lw=7, solid_capstyle="butt", zorder=4)
        ax.text(2.4, yi + h / 2, "0.0", va="center", ha="left", fontsize=14, color=INK, fontweight="bold", zorder=4, bbox=pad)
    else:
        ax.text(OURS[c] + 1.0, yi + h / 2, f"{OURS[c]:.1f}", va="center", ha="left", fontsize=14, color=INK, zorder=4, bbox=pad)
    ax.text(PAPER[c] + 1.0, yi - h / 2, f"{PAPER[c]:.1f}", va="center", ha="left", fontsize=14, color=MUTED, zorder=4, bbox=pad)
    edge = max(OURS[c], PAPER[c]) + 7.5
    weight = "bold" if gap[c] >= 20 else "normal"
    ax.text(edge, yi, f"gap {gap[c]:.1f}", va="center", ha="left", fontsize=14, color=INK, fontweight=weight, zorder=4, bbox=pad)

# the headline numbers, drawn as lines, and the rare-class handoff to the next-steps slide
ax.axvline(OURS_MEAN, color=INK, ls="--", lw=1.4, zorder=1)
ax.axvline(PAPER_MEAN, color=MUTED, ls=":", lw=1.6, zorder=1)
top = y[0] + 0.72
ax.text(OURS_MEAN, top, f"ours {OURS_MEAN}", ha="center", va="bottom", fontsize=13, color=INK, bbox=pad)
ax.text(PAPER_MEAN, top, f"paper {PAPER_MEAN}", ha="center", va="bottom", fontsize=13, color=MUTED, bbox=pad)
y_fil = y[order.index("filament")]
ax.text(PAPER["filament"] + 7.5 + 12.5, y_fil - h / 2, FILAMENT_NOTE, va="center", ha="left", fontsize=12.5, color=MUTED, style="italic", zorder=4, bbox=pad)

ax.set_yticks(y, [LABEL[c] for c in order])
ax.set_xlim(0, 100)
ax.set_ylim(-0.75, y[0] + 1.1)
ax.set_xlabel("IoU (%)")
ax.grid(axis="x", color="#ececec", zorder=0)
ax.set_axisbelow(True)
ax.text(
    0.985, 0.02,
    "upper bar: ours, val, half resolution\nlower bar: paper, MiniNet-v2 HYPER, Table IV",
    transform=ax.transAxes, ha="right", va="bottom", fontsize=13, color=INK, linespacing=1.5,
)
ax.set_title(
    f"Per-class IoU: ours {OURS_MEAN} vs paper {PAPER_MEAN}. Filament and cardboard are {gap_share:.0%} of the gap.",
    loc="left", color=INK, pad=14,
)
fig.subplots_adjust(left=0.1, right=0.98, top=0.9, bottom=0.2)
fig.text(
    0.01, 0.015,
    "Ours: MiniNetSeg, 571,879 params; half resolution for cube and mask; 30 epochs; val split, best epoch 29; one run.\n"
    "Paper: full resolution, 200 epochs. Background IoU 86.8 is not in the foreground mean.",
    fontsize=11.5, color=MUTED, linespacing=1.5,
)
save(fig, "per_class_iou")
print("gap (points):", {LABEL[c]: round(gap[c], 1) for c in order})
print("filament + cardboard share of the gap:", round(gap_share, 2))


## B. Architecture strip, current vs goal (slide 8)

Shapes per stage at a 256 input, from `src/sw/models/transformer.py`. Spectral block and spectral pool landed in PR #12 (`ResidualSelfAttentionBlock` across the 14 band groups, then a mean over groups). Spatial PE, the spatial blocks, and the seg head still raise `NotImplementedError`.

Status is box style plus a word: solid is merged or done, dashed outline is a stub. Each box names its lecture tie-in (patch and position embedding, self-attention from our own `nn.Linear` Q K V and softmax, the segmentation head, mIoU), as the ground rules ask on every slide. The brackets mark what each Phase 2 ablation row removes, which is how the diagram ties back to the hypothesis on slide 5. The subtitle carries "from scratch, no pretrained weights". The editable version for the slide is drawn in Slides from this same table; this cell is the fallback and writes an SVG too.


In [ ]:
STAGES = [
    # title, lecture tie-in, detail, output shape, params, status
    ("Input cube", "", "loader: uint16 scaled to [0, 1]", "(B, 224, 256, 256)", "0", "done"),
    (
        "Band tokenizer",
        "L1: patch + position embedding",
        "14 groups x 16 bands\nshared Conv2d 16 to 64, k8 s8\nLayerNorm, group emb. 14 x 64",
        "(B, 1024, 14, 64)",
        "66,624",
        "merged, PR #9",
    ),
    (
        "Spectral block x2",
        "L1: QKV from nn.Linear + softmax",
        "attention over the 14 band\ngroups, inside each patch",
        "(B, 1024, 14, 64)",
        "49,984 each",
        "merged, PR #12",
    ),
    ("Spectral pool", "", "mean over the 14 groups", "(B, 1024, 64)", "0", "merged, PR #12"),
    ("Spatial PE", "L1: position embedding", "fixed 2D sin-cos\non the 32 x 32 grid", "(B, 1024, 64)", "0", "stub"),
    (
        "Spatial block x6",
        "L1: QKV from nn.Linear + softmax",
        "attention over the\n1,024 patches",
        "(B, 1024, 64)",
        "49,984 each",
        "stub",
    ),
    ("Seg head", "L9: segmentation head", "(B, 64, 32, 32) to 7 classes\nupsample x8", "(B, 7, 256, 256)", "~0.5k (1x1) to ~21k", "stub"),
    (
        "Loss + metric",
        "L9: IoU and mIoU",
        "CE with paper-scheme weights\non the HSI counts;\nFG mIoU over classes 1 to 6",
        "scalar",
        "0",
        "done (harness)",
    ),
]
# Phase 2 ablation rows, as brackets over the stages each row removes: (row, first col, last col, text)
ABLATION = [
    (0, 2, 2, "Phase 2, spatial-only row: drops this"),
    (1, 0, 1, "Phase 2, spectral-only row: drops these"),
]
PER_BLOCK, N_BLOCKS, TOKENIZER = 49_984, 8, 66_624
assert TOKENIZER + N_BLOCKS * PER_BLOCK == 466_496


def kind_of(status):
    if status.startswith("stub"):
        return "stub"
    if status.startswith("in review"):
        return "review"
    return "done"


def draw_box(x, y, w, h, status, rounding):
    """Solid = merged or done, hatched = in review, dashed outline = stub. Style plus a word, never colour alone."""
    kind = kind_of(status)
    style = f"round,pad=0,rounding_size={rounding}"
    if kind == "review":
        ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle=style, facecolor="white", edgecolor="#a6a6a6", hatch="///", linewidth=0))
        ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle=style, facecolor="none", edgecolor=INK, linewidth=1.4))
    elif kind == "stub":
        ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle=style, facecolor="white", edgecolor=INK, linewidth=1.5, linestyle=(0, (4, 3))))
    else:
        ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle=style, facecolor="#ececec", edgecolor=INK, linewidth=1.4))


def text_bg(status):
    """A pad behind box text so a hatch never runs through the words."""
    return dict(facecolor="#ececec" if kind_of(status) == "done" else "white", edgecolor="none", pad=1.6)


W, H, STEP, X0 = 0.86, 0.78, 0.98, 0.09
ROW_Y = {0: 1.34, 1: 0.34}  # bottom of each row
fig, ax = plt.subplots(figsize=SLIDE)
fig.subplots_adjust(left=0, right=1, bottom=0, top=1)  # the axes are the whole slide, so 1 x unit = 3.33 in
ax.set_xlim(0, 4)
ax.set_ylim(0, 2.68)
ax.axis("off")
ax.text(0.03, 2.66, "Band-token transformer, spectral then spatial", fontsize=17, fontweight="bold", va="top", color=INK)
SUBTITLE = [
    "From scratch, no pretrained weights, about 0.49M params (designed).",
    "Merged through the spectral pool; the three dashed stages are stubs, so there is no transformer score yet.",
    "Hypothesis: competitive with the hyperspectral bars 52.8 / 54.3 at this size.",
    "The Phase 2 ablation (brackets) tests whether attention over bands earns its cost.",
]
for k, line in enumerate(SUBTITLE):
    ax.text(0.03, 2.55 - 0.065 * k, line, fontsize=11.5, va="top", color=MUTED)

for i, (title, lecture, detail, shape, params, status) in enumerate(STAGES):
    row, col = divmod(i, 4)
    x0, y0 = X0 + col * STEP, ROW_Y[row]
    draw_box(x0, y0, W, H, status, 0.03)
    cx, bg = x0 + W / 2, text_bg(status)
    ax.text(cx, y0 + H - 0.065, title, ha="center", va="top", fontsize=13, fontweight="bold", color=INK, bbox=bg)
    if lecture:
        ax.text(cx, y0 + H - 0.145, lecture, ha="center", va="top", fontsize=9, style="italic", color=MUTED, bbox=bg)
    ax.text(cx, y0 + H - 0.215, detail, ha="center", va="top", fontsize=10, color=INK, linespacing=1.22, bbox=bg)
    ax.text(cx, y0 + 0.215, shape, ha="center", va="center", fontsize=10, family="monospace", color=INK, bbox=bg)
    ax.text(cx, y0 + 0.135, f"params {params}", ha="center", va="center", fontsize=10, color=MUTED, bbox=bg)
    ax.text(cx, y0 + 0.055, status, ha="center", va="center", fontsize=10.5, style="italic", color=INK, bbox=bg)
    if col < 3:  # arrow to the next box, in the gap between the detail text and the shape line
        yc = y0 + 0.32
        ax.annotate("", xy=(x0 + STEP, yc), xytext=(x0 + W, yc), arrowprops=dict(arrowstyle="-|>", color=INK, lw=1.2))

# row 1 ends at the spectral pool; the connector goes down, left along the margin, and into the spatial PE box
pool_x = X0 + 3 * STEP + W / 2
mid_y = ROW_Y[0] - 0.08
row2_mid = ROW_Y[1] + H / 2
ax.plot([pool_x, pool_x, 0.035, 0.035], [ROW_Y[0], mid_y, mid_y, row2_mid], color=INK, lw=1.2)
ax.annotate("", xy=(X0, row2_mid), xytext=(0.035, row2_mid), arrowprops=dict(arrowstyle="-|>", color=INK, lw=1.2))

# ablation brackets above the stages each Phase 2 row removes
for row, c0, c1, text in ABLATION:
    xa, xb = X0 + c0 * STEP, X0 + c1 * STEP + W
    yb = ROW_Y[row] + H + 0.035
    ax.plot([xa, xa, xb, xb], [yb - 0.02, yb, yb, yb - 0.02], color=INK, lw=1.1)
    ax.text((xa + xb) / 2, yb + 0.02, text, ha="center", va="bottom", fontsize=10.5, color=INK)

legend_items = [("merged or done", "done"), ("stub, still to do", "stub")]
for k, (word, status) in enumerate(legend_items):
    draw_box(X0 + 1.15 * k, 0.215, 0.16, 0.07, status, 0.015)
    ax.text(X0 + 1.15 * k + 0.20, 0.25, word, va="center", fontsize=11.5, color=INK)
ax.text(
    X0, 0.15,
    f"Designed, not yet measured: {TOKENIZER:,} + {N_BLOCKS} x {PER_BLOCK:,} = {TOKENIZER + N_BLOCKS * PER_BLOCK:,} + head, "
    "about 0.47M (1x1 head) to 0.49M (recipe head).",
    va="center", fontsize=10.5, color=INK,
)
ax.text(
    X0, 0.09,
    "Half resolution (--downsample 2): 16 x 16 grid, 256 tokens.   Next: spatial PE, spatial blocks, seg head; "
    "then the Phase 2 ablation; Phase 3 INT8 is planned.",
    va="center", fontsize=10.5, color=MUTED,
)
ax.text(X0, 0.03, "L1, L9: lecture tie-ins (ViT components; segmentation and mIoU).", va="center", fontsize=10.5, color=MUTED)
save(fig, "architecture", svg=True)


## C. One cube: RGB frame, false colour with label outlines, mask, spectrum (slide 6 or 3)

Needs `SW_ROOT`. Scans every train mask (no cube reads) and keeps the earliest cube that contains filament and the most classes. The stem is printed on the figure.

If the release ships an `rgb/train` folder next to `hyper/`, the first panel is that RGB frame: the only honest way to show what a colour camera sees, since all 224 bands are infrared. Without it the panel is skipped. The false-colour panel carries the label outlines, so the transferred masks can be judged against the scene (backup slide, 79.6). The spectrum panel names the closest pair of class means in this cube, which is where a model confuses classes.


In [ ]:
if not SW_ROOT:
    print("SW_ROOT is not set; skipping cell C.")
else:
    label_dir, cube_dir = train_dirs()
    stem = choose_example_stem(label_dir)
    cube_path = cube_dir / f"{stem}.tiff"
    if not cube_path.is_file():
        raise FileNotFoundError(f"missing {cube_path}")
    cube_np = load_cube(cube_path)
    mask_np = load_mask(label_dir / f"{stem}.png")
    present = [c for c in range(len(CLASS_NAMES)) if (mask_np == c).any()]
    counts = class_counts(mask_np)
    rgb_path = find_rgb(stem)
    print(f"stem {stem}: {[CLASS_NAMES[c] for c in present]}; rgb frame: {rgb_path or 'none in the release'}")

    false_colour = cube_np[list(FALSE_COLOUR_BANDS)].transpose(1, 2, 0).copy()
    for k in range(3):
        lo, hi = np.percentile(false_colour[..., k], [2, 98])
        false_colour[..., k] = np.clip((false_colour[..., k] - lo) / (hi - lo + 1e-6), 0, 1)
    colour = np.zeros((*mask_np.shape, 3))
    for c, name in enumerate(CLASS_NAMES):
        colour[mask_np == c] = COLOR[name]

    # the class mean spectra, and the closest pair: that pair is where a model will confuse classes
    bands = np.arange(N_BANDS)
    spectra = {c: cube_np[:, mask_np == c].mean(axis=1) for c in present}
    closest = min(
        ((a, b) for i, a in enumerate(present) for b in present[i + 1 :]),
        key=lambda ab: float(np.abs(spectra[ab[0]] - spectra[ab[1]]).mean()),
    )
    closest_gap = float(np.abs(spectra[closest[0]] - spectra[closest[1]]).mean())

    n_img = 3 if rgb_path else 2
    fig, axes = plt.subplots(
        1, n_img + 1, figsize=(SLIDE[0], 5.2), gridspec_kw={"width_ratios": [1] * n_img + [1.5]}, constrained_layout=True
    )
    k = 0
    if rgb_path:
        from PIL import Image

        with Image.open(rgb_path) as img:
            axes[k].imshow(np.array(img.convert("RGB")))
        axes[k].set_title("what a colour camera sees\n(the release's RGB frame)", fontsize=13)
        axes[k].axis("off")
        k += 1
    axes[k].imshow(false_colour)
    for c in present:
        if c:  # outline each object class on the false colour, so the transferred labels can be judged
            axes[k].contour((mask_np == c).astype(float), levels=[0.5], colors=[COLOR[CLASS_NAMES[c]]], linewidths=1.5)
    axes[k].set_title("three infrared bands as false colour\nwith the label outlines", fontsize=13)
    axes[k].axis("off")
    k += 1
    axes[k].imshow(colour)
    axes[k].set_title(f"labels_hyper_lt mask\nfilament: the thin green curve, {int(counts[5]):,} px", fontsize=13)
    axes[k].axis("off")
    axes[k].legend(
        handles=[Patch(facecolor=COLOR[n], edgecolor="0.35", label=f"{i} {LABEL[n]}") for i, n in enumerate(CLASS_NAMES)],
        loc="upper center", bbox_to_anchor=(0.5, -0.02), ncol=4, frameon=False, fontsize=11,
    )
    k += 1
    ax = axes[k]
    for c in present:
        draw_line(ax, bands, spectra[c], CLASS_NAMES[c], label=f"{LABEL[CLASS_NAMES[c]]}  {counts[c]:,} px")
    ax.set_xlim(0, N_BANDS - 1)
    ax.set_ylim(0, 1)
    ax.set_xlabel("band index")
    ax.set_ylabel("scaled value (uint16 / 65535)")
    ax.secondary_xaxis("top", functions=(nominal_nm, band_of_nm)).set_xlabel("nominal nm, even spacing assumed", fontsize=12)
    ax.grid(color="#f0f0f0")
    ax.set_axisbelow(True)
    ax.legend(frameon=False, fontsize=12, loc="upper left", borderaxespad=0.2)
    ax.set_title("mean spectrum per class in this cube", fontsize=13)
    ax.text(
        0.98, 0.03,
        f"closest pair: {LABEL[CLASS_NAMES[closest[0]]]} and {LABEL[CLASS_NAMES[closest[1]]]}\n"
        f"mean gap {closest_gap:.3f}; that is where classes get confused",
        transform=ax.transAxes, ha="right", va="bottom", fontsize=11.5, color=INK, linespacing=1.4,
        bbox=dict(facecolor="white", edgecolor="none", pad=2, alpha=0.9),
    )
    named = ", ".join(LABEL[CLASS_NAMES[c]] for c in present if c)
    fig.suptitle(f"One train cube, {stem}: {named}", fontsize=16, color=INK, ha="left", x=0.0)
    nm = ", ".join(f"{v:.0f}" for v in nominal_nm(FALSE_COLOUR_BANDS))
    fig.text(
        0.0, -0.04,
        f"False colour: bands {FALSE_COLOUR_BANDS}, about {nm} nm nominal, display-stretched; the cube holds no visible light. "
        "Outlines are the transferred labels (paper Table II: 79.6 mIoU against manual labels).",
        fontsize=11, color=MUTED,
    )
    save(fig, "data_example")


## D. Spectral shape per material (slide 3 or 6)

Needs `SW_ROOT`. For each object class, take the 8 train cubes with the most pixels of that class (every train mask is scanned; only those cubes are read). Each pixel's spectrum is divided by its own mean before averaging, so brightness from illumination and thickness drops out and only the shape stays. That is what slide 3 needs: evidence that look-alike materials separate by spectrum, which raw brightness hides.

Six small multiples, one object class highlighted per panel with its plus or minus 1 sd band across cubes, the other classes in grey, background dotted. Identity comes from the panel, not from colour alone. The x axis is nominal nm with the even-spacing caveat. A second file, `band_signatures_raw.png`, keeps the raw brightness version for the backup. The class pixel share is no longer drawn here; slide 6 states it.


In [ ]:
if not SW_ROOT:
    print("SW_ROOT is not set; skipping cell D.")
else:
    label_dir, cube_dir = train_dirs()
    chosen = choose_class_cubes(label_dir, PER_CLASS)
    union = list(dict.fromkeys(stem for stems in chosen.values() for stem in stems))
    missing = [stem for stem in union if not (cube_dir / f"{stem}.tiff").is_file()]
    if missing:
        raise FileNotFoundError(f"{len(missing)} cubes missing under {cube_dir}, first {missing[0]}")
    print(f"{len(union)} cubes for the {PER_CLASS} richest masks of each object class")

    # per cube and class: the mean raw spectrum, and the mean shape (each pixel divided by its own mean)
    names_all = (*chosen, "background")
    raw_means = {name: [] for name in names_all}
    shape_means = {name: [] for name in names_all}
    pixels = {name: 0 for name in names_all}
    for stem in union:
        cube = load_cube(cube_dir / f"{stem}.tiff")
        mask = load_mask(label_dir / f"{stem}.png")
        names = {name for name, stems in chosen.items() if stem in stems}
        names.add("background")
        for name in names:
            sel = mask == CLASS_NAMES.index(name)
            if not np.any(sel):
                continue
            pix = cube[:, sel]
            pixels[name] += int(sel.sum())
            raw_means[name].append(pix.mean(axis=1))
            shape_means[name].append(shape_only(pix).mean(axis=1))

    bands = np.arange(N_BANDS)
    nm_ticks = [0, 56, 112, 168, 223]

    def panel_grid(means, ylabel, title, subtitle, fname, ylim):
        """Six panels, one object class highlighted per panel, the other classes in grey. Identity by panel, not colour."""
        stats = {name: (np.stack(v).mean(axis=0), np.stack(v).std(axis=0)) for name, v in means.items() if v}
        fig, axes = plt.subplots(2, 3, figsize=SLIDE, sharex=True, sharey=True, constrained_layout=True)
        for ax, name in zip(axes.ravel(), OBJECT_CLASSES):
            for other in OBJECT_CLASSES:
                if other != name and other in stats:
                    ax.plot(bands, stats[other][0], color="0.78", lw=1.1, zorder=1)
            if "background" in stats:
                ax.plot(bands, stats["background"][0], color="0.45", lw=1.4, ls=":", zorder=2)
            mean, std = stats[name]
            ax.fill_between(bands, mean - std, mean + std, color=COLOR[name], alpha=0.22, linewidth=0, zorder=2)
            draw_line(ax, bands, mean, name, lw=2.4)
            share = pixels[name] / HYPER_LT_TRAIN_PIXEL_COUNTS[CLASS_NAMES.index(name)]
            ax.set_title(f"{LABEL[name]}", fontsize=15, loc="left", pad=6)
            ax.text(
                0.98, 0.95, f"{len(means[name])} cubes, {share:.0%} of its train pixels",
                transform=ax.transAxes, ha="right", va="top", fontsize=11, color=MUTED,
            )
            ax.grid(color="#f0f0f0")
            ax.set_axisbelow(True)
            ax.set_xlim(0, N_BANDS - 1)
            ax.set_ylim(*ylim)
            ax.set_xticks(nm_ticks, [f"{nominal_nm(b):.0f}" for b in nm_ticks])
        for ax in axes[1]:
            ax.set_xlabel("nominal nm (band 0 to 223, even spacing assumed)", fontsize=12)
        for ax in axes[:, 0]:
            ax.set_ylabel(ylabel, fontsize=12)
        fig.suptitle(f"{title}\n{subtitle}", fontsize=15, ha="left", x=0.01, color=INK)
        save(fig, fname)

    sub = (
        f"Each panel: the {PER_CLASS} train cubes with the most pixels of that class, mean of cube means, band = 1 sd across cubes. "
        "Grey: the other classes. Dotted: background in the same cubes."
    )
    panel_grid(
        shape_means,
        "pixel / its own mean",
        "Spectral shape per material, brightness removed",
        sub,
        "band_signatures",
        (0.3, 1.7),
    )
    panel_grid(
        raw_means,
        "scaled value (uint16 / 65535)",
        "Raw spectrum per material (brightness kept)",
        sub,
        "band_signatures_raw",
        (0.0, 1.0),
    )
    for name in names_all:
        total = HYPER_LT_TRAIN_PIXEL_COUNTS[CLASS_NAMES.index(name)]
        print(f"{LABEL[name]:12} cubes {len(raw_means[name]):2}  pixels {pixels[name]:8}  share {pixels[name] / total:.1%}")


## G. Training curves from `metrics.csv` (Nov 5)

The harness writes `epoch, train_loss, val_fg_miou, steps, iou_<class>` per epoch. Copy the CNN run's file to `docs/metrics/cnn_ds2.csv` (a few KB; `docs/runs/` would be caught by the `runs/` ignore rule) and point the transformer entry at its run folder. Entries whose file is missing are skipped.


In [ ]:
import csv

RUNS = {
    "CNN control, cnn_ds2": REPO / "docs" / "metrics" / "cnn_ds2.csv",
    "band-token transformer": REPO / "runs" / "transformer" / "metrics.csv",
}
found = {name: path for name, path in RUNS.items() if path.is_file()}
if not found:
    print("No metrics file found. Copy cnn_ds2/metrics.csv to docs/metrics/cnn_ds2.csv, or edit RUNS.")
else:
    fig, (ax_l, ax_m) = plt.subplots(1, 2, figsize=(12.2, 4.8), constrained_layout=True)
    styles = [("-", INK), ("--", "#555555"), (":", MUTED)]
    for (name, path), (ls, color) in zip(found.items(), styles):
        with path.open(newline="") as handle:
            rows = list(csv.DictReader(handle))
        epochs = np.array([int(r["epoch"]) for r in rows])
        loss = np.array([float(r["train_loss"]) for r in rows])
        miou = np.array([float(r["val_fg_miou"]) * 100 if r["val_fg_miou"] else np.nan for r in rows])
        ax_l.plot(epochs, loss, ls, color=color, lw=2.2, label=name)
        ax_m.plot(epochs, miou, ls, color=color, lw=2.2, label=name)
        best = int(np.nanargmax(miou))
        ax_m.scatter([epochs[best]], [miou[best]], s=42, color=color, zorder=3)
        ax_m.annotate(
            f"{miou[best]:.1f}  epoch {epochs[best]}",
            xy=(epochs[best], miou[best]),
            xytext=(0, 8),
            textcoords="offset points",
            ha="center",
            fontsize=10,
            color=INK,
        )
    ax_l.set_xlabel("epoch")
    ax_l.set_ylabel("train loss")
    ax_l.set_title("Train loss", loc="left")
    ax_l.grid(color="#f0f0f0")
    ax_l.legend(frameon=False, loc="upper right")
    ax_m.set_xlabel("epoch")
    ax_m.set_ylabel("val FG mIoU (%)")
    ax_m.set_title("Val foreground mIoU", loc="left")
    ax_m.grid(color="#f0f0f0")
    ax_m.set_ylim(0, 70)
    ax_m.axhline(52.8, color=MUTED, ls=":", lw=1)
    ax_m.axhline(54.3, color="#b0b0b0", ls=":", lw=1)
    ax_m.annotate(
        "paper MiniNet-v2 52.8\nSegFormer-B0 54.3\nboth full res, 200 epochs",
        xy=(0.98, 54.3),
        xycoords=("axes fraction", "data"),
        ha="right",
        va="bottom",
        fontsize=9,
        color=MUTED,
    )
    ax_m.legend(frameon=False, loc="lower right")
    save(fig, "training_curves")
